**Objective:** Compare Prophet, LSTM, and ARIMA forecasts against real January 2026 F42119 data to determine the best performing model.

**This notebook is unique — most forecasting projects never get to validate against real future data. We have actual January 2026 sales data to validate all 3 models.**

**Models compared:**
- Facebook Prophet (classical decomposition)
- Stacked LSTM (deep learning)
- Auto ARIMA/SARIMA (statistical)
- Ensemble (average of all 3)

**Validation data:** `jan_raw_data.csv` — real F42119 extract for January 2026

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print('Libraries loaded successfully')

### Load Jan 2026 Actual Data

In [ ]:
# Load January 2026 raw data
df_jan = pd.read_csv('jan_raw_data.csv', low_memory=False)

print(f'Raw Jan 2026 rows    : {len(df_jan):,}')
print(f'Raw Jan 2026 columns : {df_jan.shape[1]:,}')

# Strip column name whitespace
df_jan.columns = df_jan.columns.str.strip()

# Select relevant columns
COLS = [
    'G/L Date',
    'Quantity Shipped',
    'Extended Price',
    'Ln Ty',
    'Short Item No',
    'Description',
    'Business Unit',
    'Order Number'
]
df_jan = df_jan[COLS].copy()
df_jan.columns = [
    'gl_date', 'qty_shipped', 'extended_price',
    'line_type', 'item_no', 'description',
    'business_unit', 'order_number'
]

# Fix data types (JDE exports numbers as comma-formatted strings)
df_jan['qty_shipped']    = pd.to_numeric(df_jan['qty_shipped'].astype(str).str.replace(',', ''), errors='coerce')
df_jan['extended_price'] = pd.to_numeric(df_jan['extended_price'].astype(str).str.replace(',', ''), errors='coerce')

# Drop nulls
df_jan = df_jan[df_jan['extended_price'].notna()].reset_index(drop=True)

# Parse date
df_jan['gl_date'] = pd.to_datetime(df_jan['gl_date'], errors='coerce')

# Apply JDE filters
df_jan = df_jan[df_jan['line_type'].str.strip() == 'S']
df_jan = df_jan[df_jan['qty_shipped'] > 0]
df_jan = df_jan[df_jan['extended_price'] > 0]
df_jan = df_jan[df_jan['gl_date'].notna()]

print(f'Clean Jan 2026 rows  : {len(df_jan):,}')
print(f'Date range           : {df_jan["gl_date"].min().date()} to {df_jan["gl_date"].max().date()}')

### Extract the Actual Qty & Rev for Jan 2026

In [ ]:
# Aggregate to get Jan 2026 totals
actual_jan26_qty = df_jan['qty_shipped'].sum()
actual_jan26_rev = df_jan['extended_price'].sum()
actual_jan26_orders = df_jan['order_number'].nunique()

print('=' * 55)
print('   ACTUAL JANUARY 2026 — F42119 DATA')
print('=' * 55)
print(f'Total Quantity Shipped : {actual_jan26_qty:,.0f} units')
print(f'Total Revenue          : LAK {actual_jan26_rev:,.0f}')
print(f'Total Revenue          : LAK {actual_jan26_rev/1e9:.2f} Billion')
print(f'Unique Orders          : {actual_jan26_orders:,}')

### Load All Models Forecasts Data

In [ ]:
# === FORECAST VALUES FROM EACH MODEL ===
# (These come from Notebooks 03, 04, 05)

forecasts = {
    'Prophet': {
        'qty': 843744,
        'rev': 240256227466
    },
    'LSTM': {
        'qty': 500682,
        'rev': 138893950976
    },
    'ARIMA': {
        'qty': 671544,
        'rev': 198275932379
    }
}

# Ensemble = simple average of all 3
forecasts['Ensemble'] = {
    'qty': np.mean([forecasts[m]['qty'] for m in ['Prophet', 'LSTM', 'ARIMA']]),
    'rev': np.mean([forecasts[m]['rev'] for m in ['Prophet', 'LSTM', 'ARIMA']])
}

print('Forecast values loaded:')
for model, vals in forecasts.items():
    print(f'  {model:10s} → Qty: {vals["qty"]:>12,.0f} units | Rev: LAK {vals["rev"]/1e9:>8.2f}B')

### FINAL Validation - Forecast vs Actual

In [ ]:
def mape_single(actual, predicted):
    return abs((predicted - actual) / actual) * 100

print('=' * 70)
print('   JANUARY 2026 FORECAST vs ACTUAL — FINAL VALIDATION')
print('=' * 70)
print(f'Actual Quantity : {actual_jan26_qty:>15,.0f} units')
print(f'Actual Revenue  : LAK {actual_jan26_rev:>20,.0f}')
print()
print(f'{"Model":<12} {"Pred Qty":>12} {"Qty Err%":>10} {"Pred Rev (B)":>14} {"Rev Err%":>10}')
print('-' * 70)

results = {}
for model, vals in forecasts.items():
    qty_err = (vals['qty'] - actual_jan26_qty) / actual_jan26_qty * 100
    rev_err = (vals['rev'] - actual_jan26_rev) / actual_jan26_rev * 100
    qty_mape = mape_single(actual_jan26_qty, vals['qty'])
    rev_mape = mape_single(actual_jan26_rev, vals['rev'])
    results[model] = {
        'qty_pred' : vals['qty'],
        'rev_pred' : vals['rev'],
        'qty_err'  : qty_err,
        'rev_err'  : rev_err,
        'qty_mape' : qty_mape,
        'rev_mape' : rev_mape,
        'avg_mape' : (qty_mape + rev_mape) / 2
    }
    print(f'{model:<12} {vals["qty"]:>12,.0f} {qty_err:>+9.1f}% {vals["rev"]/1e9:>13.2f}B {rev_err:>+9.1f}%')

print()
# Find winner
winner = min(results, key=lambda x: results[x]['avg_mape'])
print(f'WINNER (lowest avg MAPE): {winner} — {results[winner]["avg_mape"]:.2f}% average error')

### Visualization of Plot - Actual vs Forecast

In [ ]:
models      = list(forecasts.keys())
colors      = ['#2196F3', '#FF9800', '#4CAF50', '#9C27B0']
qty_preds   = [forecasts[m]['qty'] for m in models]
rev_preds   = [forecasts[m]['rev'] for m in models]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('January 2026 — All Model Forecasts vs Actual', fontsize=15, fontweight='bold')

x = np.arange(len(models))
width = 0.5

# --- Quantity ---
bars1 = axes[0].bar(x, qty_preds, width, color=colors, alpha=0.85, label='Predicted')
axes[0].axhline(actual_jan26_qty, color='red', linewidth=2.5, linestyle='--', label=f'Actual: {actual_jan26_qty:,.0f}')
axes[0].set_title('Quantity Shipped — Forecast vs Actual', fontweight='bold')
axes[0].set_ylabel('Quantity (Units)')
axes[0].set_xticks(x)
axes[0].set_xticklabels(models, fontsize=11)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:,.0f}'))
axes[0].legend()
for bar, model in zip(bars1, models):
    err = results[model]['qty_err']
    mape = results[model]['qty_mape']
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height()*1.01,
                 f'{err:+.1f}%\nMAPE:{mape:.1f}%', ha='center', fontsize=9, fontweight='bold')

# --- Revenue ---
bars2 = axes[1].bar(x, [r/1e9 for r in rev_preds], width, color=colors, alpha=0.85)
axes[1].axhline(actual_jan26_rev/1e9, color='red', linewidth=2.5, linestyle='--',
                label=f'Actual: LAK {actual_jan26_rev/1e9:.2f}B')
axes[1].set_title('Revenue (LAK) — Forecast vs Actual', fontweight='bold')
axes[1].set_ylabel('Revenue (LAK Billions)')
axes[1].set_xticks(x)
axes[1].set_xticklabels(models, fontsize=11)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.0f}B'))
axes[1].legend()
for bar, model in zip(bars2, models):
    err = results[model]['rev_err']
    mape = results[model]['rev_mape']
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height()*1.01,
                 f'{err:+.1f}%\nMAPE:{mape:.1f}%', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('reports/13_final_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to reports/13_final_comparison.png')

### Scorecard

In [ ]:
# Test set MAPEs from previous notebooks
test_mapes = {
    'Prophet' : {'qty': 19.08, 'rev': 11.92},
    'LSTM'    : {'qty': 5.86,  'rev': 9.22},
    'ARIMA'   : {'qty': 15.19, 'rev': 11.93},
    'Ensemble': {'qty': None,  'rev': None}
}

print('=' * 75)
print('   COMPLETE MODEL SCORECARD')
print('=' * 75)
print(f'{"Model":<12} {"Test Qty MAPE":>15} {"Test Rev MAPE":>15} {"Jan26 Qty MAPE":>16} {"Jan26 Rev MAPE":>16}')
print('-' * 75)
for model in ['Prophet', 'LSTM', 'ARIMA', 'Ensemble']:
    tq = f"{test_mapes[model]['qty']:.2f}%" if test_mapes[model]['qty'] else 'N/A'
    tr = f"{test_mapes[model]['rev']:.2f}%" if test_mapes[model]['rev'] else 'N/A'
    jq = f"{results[model]['qty_mape']:.2f}%"
    jr = f"{results[model]['rev_mape']:.2f}%"
    print(f'{model:<12} {tq:>15} {tr:>15} {jq:>16} {jr:>16}')

print()
print(f'BEST MODEL (Jan 2026 Avg MAPE):')
for model in ['Prophet', 'LSTM', 'ARIMA', 'Ensemble']:
    print(f'  {model:<12}: {results[model]["avg_mape"]:.2f}% avg MAPE')

### MAPE Comparision

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('MAPE Comparison: Test Set vs January 2026 Actual', fontsize=14, fontweight='bold')

model_labels = ['Prophet', 'LSTM', 'ARIMA', 'Ensemble']
x = np.arange(len(model_labels))
width = 0.35

# Quantity MAPE
test_qty_mapes = [19.08, 5.86, 15.19, None]
jan_qty_mapes  = [results[m]['qty_mape'] for m in model_labels]

b1 = axes[0].bar(x - width/2, [v if v else 0 for v in test_qty_mapes],
                  width, label='Test Set (Oct-Dec 2025)', color='steelblue', alpha=0.8)
b2 = axes[0].bar(x + width/2, jan_qty_mapes,
                  width, label='Jan 2026 Actual', color='darkorange', alpha=0.8)
axes[0].set_title('Quantity MAPE by Model', fontweight='bold')
axes[0].set_ylabel('MAPE (%)')
axes[0].set_xticks(x)
axes[0].set_xticklabels(model_labels)
axes[0].legend()
for bar in b1:
    if bar.get_height() > 0:
        axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                     f'{bar.get_height():.1f}%', ha='center', fontsize=9)
for bar in b2:
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                 f'{bar.get_height():.1f}%', ha='center', fontsize=9)

# Revenue MAPE
test_rev_mapes = [11.92, 9.22, 11.93, None]
jan_rev_mapes  = [results[m]['rev_mape'] for m in model_labels]

b3 = axes[1].bar(x - width/2, [v if v else 0 for v in test_rev_mapes],
                  width, label='Test Set (Oct-Dec 2025)', color='steelblue', alpha=0.8)
b4 = axes[1].bar(x + width/2, jan_rev_mapes,
                  width, label='Jan 2026 Actual', color='darkorange', alpha=0.8)
axes[1].set_title('Revenue MAPE by Model', fontweight='bold')
axes[1].set_ylabel('MAPE (%)')
axes[1].set_xticks(x)
axes[1].set_xticklabels(model_labels)
axes[1].legend()
for bar in b3:
    if bar.get_height() > 0:
        axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                     f'{bar.get_height():.1f}%', ha='center', fontsize=9)
for bar in b4:
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                 f'{bar.get_height():.1f}%', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('reports/14_mape_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to reports/14_mape_comparison.png')

### Final Recommendation

In [ ]:
best_qty   = min(results, key=lambda x: results[x]['qty_mape'])
best_rev   = min(results, key=lambda x: results[x]['rev_mape'])
best_overall = min(results, key=lambda x: results[x]['avg_mape'])

print('=' * 65)
print('   FINAL PROJECT SUMMARY — JDE F42119 DEMAND FORECASTING')
print('=' * 65)
print()
print('ACTUAL JANUARY 2026 RESULTS:')
print(f'  Quantity : {actual_jan26_qty:,.0f} units')
print(f'  Revenue  : LAK {actual_jan26_rev/1e9:.2f} Billion')
print()
print('MODEL PERFORMANCE ON JANUARY 2026:')
for model in ['Prophet', 'LSTM', 'ARIMA', 'Ensemble']:
    print(f'  {model:<12}: Qty MAPE={results[model]["qty_mape"]:.2f}%  Rev MAPE={results[model]["rev_mape"]:.2f}%  Avg={results[model]["avg_mape"]:.2f}%')
print()
print(f'BEST FOR QUANTITY : {best_qty} ({results[best_qty]["qty_mape"]:.2f}% MAPE)')
print(f'BEST FOR REVENUE  : {best_rev} ({results[best_rev]["rev_mape"]:.2f}% MAPE)')
print(f'BEST OVERALL      : {best_overall} ({results[best_overall]["avg_mape"]:.2f}% avg MAPE)')
print()
print('BUSINESS RECOMMENDATION:')
print(f'  For production deployment, recommend the {best_overall} model')
print(f'  for monthly demand forecasting of JDE F42119 sales data.')
print()
print('  Reasoning:')
print('  - ARIMA: Fast, interpretable, good for stable trend detection')
print('  - Prophet: Best for capturing seasonality and structural breaks')
print('  - LSTM: Best when recent trend patterns dominate')
print('  - Ensemble: Balances all three — most robust for production use')